In [1]:
!pip install mrjob -q
print("mrjob installed ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 6.0 MB/s eta 0:00:00
mrjob installed ✓


In [2]:
# ─── Q1: WORD COUNT ───────────────────────────────────────────
from collections import defaultdict

lines_q1 = ["hadoop is fast", "hadoop is scalable"]

# ── (A) Manual MapReduce ──
def mapper_wc(lines):
    return [(word, 1) for line in lines for word in line.split()]

def shuffle(pairs):
    grouped = defaultdict(list)
    for k, v in pairs:
        grouped[k].append(v)
    return dict(grouped)

def reducer_sum(grouped):
    return {k: sum(v) for k, v in sorted(grouped.items())}

result_q1 = reducer_sum(shuffle(mapper_wc(lines_q1)))
print("Q1 Manual →", result_q1)

# ── (B) mrjob ──
q1_input = "\n".join(lines_q1)
with open("q1_input.txt", "w") as f:
    f.write(q1_input)

Q1 Manual → {'fast': 1, 'hadoop': 2, 'is': 2, 'scalable': 1}


In [3]:
%%writefile q1_wordcount.py
from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordCount.run()

Writing q1_wordcount.py


In [4]:
!python q1_wordcount.py q1_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q1_wordcount.root.20260426.081120.756216
Running step 1 of 1...
job output is in /tmp/q1_wordcount.root.20260426.081120.756216/output
Streaming final output from /tmp/q1_wordcount.root.20260426.081120.756216/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/q1_wordcount.root.20260426.081120.756216...


In [5]:
# ─── Q2: CHARACTER COUNT (ignore spaces) ─────────────────────
text_q2 = "big data"

# ── (A) Manual ──
def mapper_cc(text):
    return [(ch, 1) for ch in text if ch != " "]

result_q2 = reducer_sum(shuffle(mapper_cc(text_q2)))
print("Q2 Manual →", result_q2)

with open("q2_input.txt", "w") as f:
    f.write(text_q2)

Q2 Manual → {'a': 2, 'b': 1, 'd': 1, 'g': 1, 'i': 1, 't': 1}


In [6]:
%%writefile q2_charcount.py
from mrjob.job import MRJob

class MRCharCount(MRJob):
    def mapper(self, _, line):
        for ch in line.strip():
            if ch != " ":
                yield ch, 1

    def reducer(self, ch, counts):
        yield ch, sum(counts)

if __name__ == "__main__":
    MRCharCount.run()

Writing q2_charcount.py


In [7]:
!python q2_charcount.py q2_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2_charcount.root.20260426.081140.826740
Running step 1 of 1...
job output is in /tmp/q2_charcount.root.20260426.081140.826740/output
Streaming final output from /tmp/q2_charcount.root.20260426.081140.826740/output...
"b"	1
"d"	1
"g"	1
"i"	1
"t"	1
"a"	2
Removing temp directory /tmp/q2_charcount.root.20260426.081140.826740...


In [8]:
# ─── Q3: AVERAGE WORD LENGTH PER WORD ────────────────────────
text_q3 = "data science data big data"

# ── (A) Manual ──
def mapper_awl(text):
    return [(word, len(word)) for word in text.split()]

def reducer_avg(grouped):
    return {k: sum(v) / len(v) for k, v in sorted(grouped.items())}

result_q3 = reducer_avg(shuffle(mapper_awl(text_q3)))
print("Q3 Manual →", result_q3)

with open("q3_input.txt", "w") as f:
    f.write(text_q3)

Q3 Manual → {'big': 3.0, 'data': 4.0, 'science': 7.0}


In [9]:
%%writefile q3_avgwordlen.py
from mrjob.job import MRJob

class MRAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word, len(word)

    def reducer(self, word, lengths):
        lengths = list(lengths)
        yield word, sum(lengths) / len(lengths)

if __name__ == "__main__":
    MRAvgWordLen.run()

Writing q3_avgwordlen.py


In [10]:
!python q3_avgwordlen.py q3_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3_avgwordlen.root.20260426.081204.100683
Running step 1 of 1...
job output is in /tmp/q3_avgwordlen.root.20260426.081204.100683/output
Streaming final output from /tmp/q3_avgwordlen.root.20260426.081204.100683/output...
"science"	7.0
"big"	3.0
"data"	4.0
Removing temp directory /tmp/q3_avgwordlen.root.20260426.081204.100683...


In [11]:
# ─── Q4: GLOBAL AVERAGE WORD LENGTH ──────────────────────────
text_q4 = "hadoop mapreduce spark"

# ── (A) Manual ──
def mapper_global_avg(text):
    return [("__global__", len(word)) for word in text.split()]

result_q4 = reducer_avg(shuffle(mapper_global_avg(text_q4)))
print("Q4 Manual → Global avg word length:", result_q4["__global__"])

with open("q4_input.txt", "w") as f:
    f.write(text_q4)

Q4 Manual → Global avg word length: 6.666666666666667


In [12]:
%%writefile q4_globalavg.py
from mrjob.job import MRJob

class MRGlobalAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield "__global__", (len(word), 1)

    def reducer(self, key, values):
        total_len, total_count = 0, 0
        for length, count in values:
            total_len += length
            total_count += count
        yield "global_avg_word_length", total_len / total_count

if __name__ == "__main__":
    MRGlobalAvgWordLen.run()

Writing q4_globalavg.py


In [13]:
!python q4_globalavg.py q4_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4_globalavg.root.20260426.081226.690190
Running step 1 of 1...
job output is in /tmp/q4_globalavg.root.20260426.081226.690190/output
Streaming final output from /tmp/q4_globalavg.root.20260426.081226.690190/output...
"global_avg_word_length"	6.666666666666667
Removing temp directory /tmp/q4_globalavg.root.20260426.081226.690190...


In [14]:
# ─── Q5: Download file from Google Drive ─────────────────────
!pip install gdown -q
!gdown 16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas -O q5_input.txt
print("File downloaded ✓")

Downloading...
From: https://drive.google.com/uc?id=16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas
To: /content/q5_input.txt
100% 5.59M/5.59M [00:00<00:00, 62.7MB/s]
File downloaded ✓


In [15]:
# ─── Q5(A): Manual — Word Count + Top 5 ──────────────────────
import re

with open("q5_input.txt", "r", errors="ignore") as f:
    raw = f.read().lower()

words = re.findall(r"[a-z]+", raw)

# Word Count
wc = defaultdict(int)
for w in words:
    wc[w] += 1

print("Top 5 most frequent words:")
for word, count in sorted(wc.items(), key=lambda x: -x[1])[:5]:
    print(f"  {word}: {count}")

# Char Count
cc = defaultdict(int)
for ch in raw:
    if ch.isalpha():
        cc[ch] += 1
print("\nChar Count (sample, a-e):", {k: cc[k] for k in sorted(cc)[:5]})

# Avg word length per word
awl = defaultdict(list)
for w in words:
    awl[w].append(len(w))
# (too many unique words to print all, show sample)
sample_awl = {k: sum(v)/len(v) for k, v in list(awl.items())[:5]}
print("\nAvg Word Length per Word (sample):", sample_awl)

# Global avg word length
total_len = sum(len(w) for w in words)
global_avg = total_len / len(words)
print(f"\nGlobal Avg Word Length: {global_avg:.4f}")

Top 5 most frequent words:
  the: 27843
  and: 26847
  i: 22538
  to: 19882
  of: 18307

Char Count (sample, a-e): {'a': 290069, 'b': 62218, 'c': 88720, 'd': 149942, 'e': 448860}

Avg Word Length per Word (sample): {'the': 3.0, 'project': 7.0, 'gutenberg': 9.0, 'ebook': 5.0, 'of': 2.0}

Global Avg Word Length: 4.0878


In [16]:
%%writefile q5_wordcount.py
from mrjob.job import MRJob
import re

class MRWordCountFile(MRJob):
    def mapper(self, _, line):
        for word in re.findall(r"[a-z]+", line.lower()):
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordCountFile.run()

Writing q5_wordcount.py


In [17]:
# Run mrjob word count on the file, then find top 5
import subprocess, json

result = subprocess.run(
    ["python", "q5_wordcount.py", "q5_input.txt"],
    capture_output=True, text=True
)

word_counts = []
for line in result.stdout.strip().split("\n"):
    if line.strip():
        word, count = line.split("\t")
        word_counts.append((word.strip('"'), int(count)))

word_counts.sort(key=lambda x: -x[1])
print("Q5 mrjob — Top 5 words:")
for w, c in word_counts[:5]:
    print(f"  {w}: {c}")

Q5 mrjob — Top 5 words:
  the: 27843
  and: 26847
  i: 22538
  to: 19882
  of: 18307


In [18]:
# ─── Q6: AVERAGE MARKS PER STUDENT ───────────────────────────
data_q6 = [("A", 80), ("B", 70), ("A", 90), ("B", 60), ("A", 100)]

# ── (A) Manual ──
def mapper_marks(data):
    return [(student, mark) for student, mark in data]

grouped_q6 = shuffle(mapper_marks(data_q6))
result_q6 = {k: sum(v)/len(v) for k, v in sorted(grouped_q6.items())}
print("Q6 Manual → Avg marks:", result_q6)

with open("q6_input.txt", "w") as f:
    for student, mark in data_q6:
        f.write(f"{student} {mark}\n")

Q6 Manual → Avg marks: {'A': 90.0, 'B': 65.0}


In [19]:
%%writefile q6_avgmarks.py
from mrjob.job import MRJob

class MRAvgMarks(MRJob):
    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], float(parts[1])

    def reducer(self, student, marks):
        marks = list(marks)
        yield student, sum(marks) / len(marks)

if __name__ == "__main__":
    MRAvgMarks.run()

Writing q6_avgmarks.py


In [20]:
!python q6_avgmarks.py q6_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6_avgmarks.root.20260426.081337.156943
Running step 1 of 1...
job output is in /tmp/q6_avgmarks.root.20260426.081337.156943/output
Streaming final output from /tmp/q6_avgmarks.root.20260426.081337.156943/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/q6_avgmarks.root.20260426.081337.156943...


In [21]:
# ─── Q7: AVG SALARY PER DEPT + HIGHEST PAID DEPT ────────────
data_q7 = [("HR",50000),("IT",70000),("HR",60000),("IT",80000)]

# ── (A) Manual ──
grouped_q7 = shuffle([(dept, sal) for dept, sal in data_q7])
avg_sal = {k: sum(v)/len(v) for k, v in sorted(grouped_q7.items())}
highest_dept = max(avg_sal, key=avg_sal.get)
print("Q7 Manual → Avg salary:", avg_sal)
print("Q7 Manual → Highest paid dept:", highest_dept, "→", avg_sal[highest_dept])

with open("q7_input.txt", "w") as f:
    for dept, sal in data_q7:
        f.write(f"{dept} {sal}\n")

Q7 Manual → Avg salary: {'HR': 55000.0, 'IT': 75000.0}
Q7 Manual → Highest paid dept: IT → 75000.0


In [22]:
%%writefile q7_avgsalary.py
from mrjob.job import MRJob

class MRAvgSalary(MRJob):
    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], float(parts[1])

    def reducer(self, dept, salaries):
        salaries = list(salaries)
        yield dept, sum(salaries) / len(salaries)

if __name__ == "__main__":
    MRAvgSalary.run()

Writing q7_avgsalary.py


In [23]:
import subprocess

res = subprocess.run(["python", "q7_avgsalary.py", "q7_input.txt"],
                     capture_output=True, text=True)
dept_avgs = {}
for line in res.stdout.strip().split("\n"):
    if line.strip():
        dept, avg = line.split("\t")
        dept_avgs[dept.strip('"')] = float(avg)

print("Q7 mrjob → Avg salaries:", dept_avgs)
print("Highest paid dept:", max(dept_avgs, key=dept_avgs.get))

Q7 mrjob → Avg salaries: {'IT': 75000.0, 'HR': 55000.0}
Highest paid dept: IT


In [24]:
# ─── Q8: AVERAGE TEMPERATURE PER CITY ────────────────────────
data_q8 = [
    ("New York", 38), ("London", 29), ("Tokyo", 35),
    ("New York", 32), ("Delhi", 45), ("Ambala", 35)
]

# ── (A) Manual ──
# Input uses comma as delimiter: "New York,38"
grouped_q8 = shuffle([(city, temp) for city, temp in data_q8])
result_q8 = {k: sum(v)/len(v) for k, v in sorted(grouped_q8.items())}
print("Q8 Manual → Avg temp:", result_q8)

with open("q8_input.txt", "w") as f:
    for city, temp in data_q8:
        f.write(f"{city},{temp}\n")

Q8 Manual → Avg temp: {'Ambala': 35.0, 'Delhi': 45.0, 'London': 29.0, 'New York': 35.0, 'Tokyo': 35.0}


In [25]:
%%writefile q8_avgtemp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):
    def mapper(self, _, line):
        line = line.strip()
        if "," in line:
            # Split on last comma to handle "New York,38"
            idx = line.rfind(",")
            city = line[:idx].strip()
            try:
                temp = float(line[idx+1:].strip())
                yield city, temp
            except ValueError:
                pass

    def reducer(self, city, temps):
        temps = list(temps)
        yield city, sum(temps) / len(temps)

if __name__ == "__main__":
    MRAvgTemp.run()

Writing q8_avgtemp.py


In [26]:
!python q8_avgtemp.py q8_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8_avgtemp.root.20260426.081436.740116
Running step 1 of 1...
job output is in /tmp/q8_avgtemp.root.20260426.081436.740116/output
Streaming final output from /tmp/q8_avgtemp.root.20260426.081436.740116/output...
"London"	29.0
"New York"	35.0
"Tokyo"	35.0
"Ambala"	35.0
"Delhi"	45.0
Removing temp directory /tmp/q8_avgtemp.root.20260426.081436.740116...


In [27]:
#Q9
# ─── CELL 1: Install & Download ───────────────────────────────
!pip install kagglehub -q

import kagglehub
import os

path = kagglehub.dataset_download("heemalichaudhari/global-land-temperatures")
print("Path to dataset files:", path)

# List available files
for f in os.listdir(path):
    print(f)

100%|██████████| 1.97M/1.97M [00:00<00:00, 105MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/heemalichaudhari/global-land-temperatures/versions/1
GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv


In [29]:
import os

# We use the By-Country file — most direct for this question
COUNTRY_FILE = os.path.join(path, "GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv")
print("Using file:", COUNTRY_FILE)

# Preview first 3 rows
with open(COUNTRY_FILE, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 2:
            break

Using file: /root/.cache/kagglehub/datasets/heemalichaudhari/global-land-temperatures/versions/1/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv
dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
1849-01-01,26.704,1.435,Abidjan,Côte D'Ivoire,5.63N,3.23W
1849-02-01,27.434,1.362,Abidjan,Côte D'Ivoire,5.63N,3.23W


In [30]:
# ─── CELL 3 (A): Manual MapReduce ─────────────────────────────
import csv
from collections import defaultdict

# ── Mapper ──
def mapper_q9(filepath):
    pairs = []
    with open(filepath, "r", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            country  = row.get("Country", "").strip()
            temp_str = row.get("AverageTemperature", "").strip()
            if country and temp_str:          # skip missing values
                try:
                    pairs.append((country, float(temp_str)))
                except ValueError:
                    pass
    return pairs

# ── Shuffle ──
def shuffle(pairs):
    grouped = defaultdict(list)
    for k, v in pairs:
        grouped[k].append(v)
    return dict(grouped)

# ── Reducer ──
def reducer_avg(grouped):
    return {k: round(sum(v) / len(v), 4) for k, v in sorted(grouped.items())}

# ── Run ──
mapped   = mapper_q9(COUNTRY_FILE)
grouped  = shuffle(mapped)
result   = reducer_avg(grouped)

print(f"Total countries processed: {len(result)}\n")
print("Top 10 Hottest Countries (by average temperature):")
for country, avg in sorted(result.items(), key=lambda x: -x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

print("\nTop 10 Coldest Countries:")
for country, avg in sorted(result.items(), key=lambda x: x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

Total countries processed: 49

Top 10 Hottest Countries (by average temperature):
  Sudan                          29.08 °C
  Vietnam                        27.19 °C
  Thailand                       27.16 °C
  Somalia                        27.15 °C
  Burma                          26.74 °C
  Indonesia                      26.66 °C
  Singapore                      26.52 °C
  Philippines                    26.45 °C
  Saudi Arabia                   26.43 °C
  Nigeria                        26.36 °C

Top 10 Coldest Countries:
  Russia                         3.96 °C
  Canada                         5.11 °C
  Chile                          5.69 °C
  Ukraine                        7.04 °C
  Germany                        8.92 °C
  United Kingdom                 9.46 °C
  France                         10.40 °C
  South Korea                    10.68 °C
  United States                  11.26 °C
  Spain                          11.45 °C


In [31]:
# ─── CELL 4 (B): mrjob — write script ─────────────────────────
%%writefile q9_country_temp.py
from mrjob.job import MRJob
import csv, io

class MRCountryAvgTemp(MRJob):

    def mapper(self, _, line):
        line = line.strip()
        if not line or line.startswith("dt"):   # skip header
            return
        try:
            row = next(csv.reader(io.StringIO(line)))
            # Columns: dt, AverageTemperature, AverageTemperatureUncertainty, Country
            if len(row) >= 4 and row[1].strip():
                country = row[3].strip()
                temp    = float(row[1].strip())
                yield country, (temp, 1)
        except (ValueError, StopIteration):
            pass

    def reducer(self, country, values):
        total, count = 0.0, 0
        for temp, cnt in values:
            total += temp
            count += cnt
        yield country, round(total / count, 4)

if __name__ == "__main__":
    MRCountryAvgTemp.run()

Writing q9_country_temp.py


In [32]:
# ─── CELL 5 (B): mrjob — run & display results ────────────────
import subprocess

res = subprocess.run(
    ["python", "q9_country_temp.py", COUNTRY_FILE],
    capture_output=True, text=True
)

# Parse output
country_avgs = {}
for line in res.stdout.strip().split("\n"):
    if "\t" in line:
        country, avg = line.split("\t")
        country_avgs[country.strip('"')] = float(avg)

print(f"mrjob processed {len(country_avgs)} countries\n")

print("Top 10 Hottest Countries:")
for country, avg in sorted(country_avgs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

print("\nTop 10 Coldest Countries:")
for country, avg in sorted(country_avgs.items(), key=lambda x: x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

mrjob processed 100 countries

Top 10 Hottest Countries:
  Umm Durman                     29.08 °C
  Madras                         28.42 °C
  Jiddah                         27.69 °C
  Ho Chi Minh City               27.19 °C
  Bangkok                        27.16 °C
  Mogadishu                      27.15 °C
  Fortaleza                      27.01 °C
  Hyderabad                      26.87 °C
  Surabaya                       26.81 °C
  Rangoon                        26.74 °C

Top 10 Coldest Countries:
  Harbin                         3.63 °C
  Saint Petersburg               3.92 °C
  Moscow                         4.00 °C
  Montreal                       4.45 °C
  Changchun                      4.92 °C
  Santiago                       5.69 °C
  Toronto                        5.77 °C
  Kiev                           7.04 °C
  Shenyang                       7.21 °C
  Taiyuan                        7.98 °C
